# Data loading

In [ ]:
import copy
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statistics import mean, stdev
from scipy.stats import norm, multivariate_normal

In [ ]:
rcupd = {
    'figure.figsize': (5, 4),
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': 'cm',
    'font.size': 12,
}
plt.rcParams.update(rcupd)

In [ ]:
data_files = [
    '2025-11-04/RERTR5_V6018G.csv',
    '2025-11-04/RERTR12_L1P755.csv',
]

# Convenience functions

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

def mod_metrics(mod, X_test, y_test):
    y_pred = mod.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(
        ' R2: ', r2, '\n',
        'RMSE: ', rmse, '\n',
        'MAE: ', mae
    )

In [ ]:
def pred_vs_actual(mod, X_test, y_test, tt):
    y_pred = mod.predict(X_test)

    plt.figure(figsize=(5,4))
    plt.rcParams.update({'font.size': 16})

    plt.scatter(y_test, y_pred, s=15)

    minv = int(min(min(y_test), min(y_pred)))
    maxv = int(max(max(y_test), max(y_pred)))
    val = list(range(minv, maxv))
    
    plt.plot(val, val, color='k', ls='--', label='y=x')

    plt.title(tt)
    plt.xlabel(r'Test data (swelling \%)')
    plt.ylabel(r'Surrogate pred. (swelling \%)')
    plt.legend()
    plt.show()

# Preprocessing

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
def load_data(fileName):
    jar = pd.read_csv(fileName)

    col_names = jar.columns[1:-2]

    X = jar.iloc[:, 1:-2].to_numpy()
    y = jar.iloc[:, -2].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=19
    )

    return X_train, X_test, y_train, y_test, col_names

In [ ]:
X_train_lo, X_test_lo, y_train_lo, y_test_lo, col_names = load_data(data_files[0])
X_train_hi, X_test_hi, y_train_hi, y_test_hi, _ = load_data(data_files[1])

In [ ]:
X_comb = np.concatenate((X_train_lo, X_train_hi), axis=0)

In [ ]:
xscaler = MinMaxScaler()
xscaler.fit(X_comb)

In [ ]:
X_train_lo = xscaler.transform(X_train_lo)
X_test_lo = xscaler.transform(X_test_lo)

X_train_hi = xscaler.transform(X_train_hi)
X_test_hi = xscaler.transform(X_test_hi)

# NN

In [ ]:
from sklearn.neural_network import MLPRegressor

In [ ]:
reg_lo = MLPRegressor(
    hidden_layer_sizes=(50, 50, 50, 50),
    alpha=0,
    random_state=37,
    max_iter=5000,
    tol=0.1
).fit(X_train_lo, y_train_lo)

mod_metrics(reg_lo, X_test_lo, y_test_lo)
pred_vs_actual(reg_lo, X_test_lo, y_test_lo, 'NN low Fd')

In [ ]:
reg_hi = MLPRegressor(
    hidden_layer_sizes=(50, 50, 50, 50),
    alpha=0,
    random_state=37,
    max_iter=5000,
    tol=0.1
).fit(X_train_hi, y_train_hi)

mod_metrics(reg_hi, X_test_hi, y_test_hi)
pred_vs_actual(reg_hi, X_test_hi, y_test_hi, 'NN hiw Fd')

# Multivariate gaussian

In [ ]:
def cr_multivar_gaussian(mu1, sig1, mu2, sig2, cov12):
    means = np.array([mu1, mu2])
    cov_matrix = np.array([[sig1**2, cov12],
                            [cov12, sig2**2]])
    mvn = multivariate_normal(mean=means, cov=cov_matrix)
    return mvn

In [ ]:
mvn = cr_multivar_gaussian(10.7, 2.64, 32, 2.64, 0)

In [ ]:
x, y = np.mgrid[0:20:0.1, 20:45:0.1]
pos = np.dstack((x, y))

fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111)
cs = ax.contourf(x, y, mvn.pdf(pos), cmap='Purples', levels=10)
plt.colorbar(cs, label='PDF')

ax.set_xlabel(r"Observed swelling at low $F_d$")
ax.set_ylabel(r"Observed swelling at high $F_d$")
plt.show()

# MCMC sampler

In [ ]:
def proposal_dist(X, sig):
    ret = []
    
    for el in X:
        prop = np.random.normal(el, sig)
        ret.append(prop)

    assert len(X) == len(ret)
    return ret

In [ ]:
def mcmc_sampler(num_param, initial_state, proposal_sig,
                 surrogates, target_mvn, num_samples):
    samples = [initial_state]
    accepted = 0

    for ii in range(num_samples):
        current_state = samples[-1]
        proposed_state = proposal_dist(current_state, proposal_sig)

        valid = True
        for xx in proposed_state:
            if xx < 0 or xx > 1:
                valid = False
                break

        fs1_curr = surrogates[0].predict([[*current_state]])[0]
        fs2_curr = surrogates[1].predict([[*current_state]])[0]
        
        fs1_prop = surrogates[0].predict([[*proposed_state]])[0]
        fs2_prop = surrogates[1].predict([[*proposed_state]])[0]
        
        acceptance_ratio = target_mvn.pdf([fs1_prop, fs2_prop]) / target_mvn.pdf([fs1_curr, fs2_curr])
        
        if valid and np.random.rand() < acceptance_ratio:
            current_state = proposed_state
            accepted += 1

        samples.append(current_state)

    print(f"Acceptance rate: {accepted / num_samples}")
    return np.array(samples)

In [ ]:
hey1 = mcmc_sampler(
    9,
    np.random.rand(9),
    0.07,
    [reg_lo, reg_hi],
    mvn,
    100000
)

In [ ]:
hey2 = mcmc_sampler(
    9,
    np.random.rand(9),
    0.07,
    [reg_lo, reg_hi],
    mvn,
    100000
)

# Trace/Hist

In [ ]:
old1 = xscaler.inverse_transform(hey1)
old2 = xscaler.inverse_transform(hey2)

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))

for i, ax in enumerate(axes.flatten()):
    cdat1 = old1[:,i]
    cavg1 = np.cumsum(cdat1) / np.arange(1, len(cdat1)+1)
    ax.plot(cdat1, lw=0.1, alpha=0.7, zorder=1)
    ax.plot(cavg1, c='k', zorder=2)
    
    cdat2 = old2[:,i]
    cavg2 = np.cumsum(cdat2) / np.arange(1, len(cdat2)+1)
    ax.plot(cdat2, ls='--', lw=0.1, alpha=0.7, zorder=1)
    ax.plot(cavg2, c='r', zorder=2)
    
    ax.set_xlabel(col_names[i])
    #ax.set_ylim([0, 1])

#fig.delaxes(axes[1,3])
fig.supylabel('Parameter values')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))

for i, ax in enumerate(axes.flatten()):
    sns.histplot(old1[:,i], ax=ax, stat='density', kde=True)
    sns.histplot(old2[:,i], ax=ax, stat='density', kde=True)
    ax.set_xlabel(col_names[i])
    ax.set_ylabel('')
    #ax.set_xlim([0, 1])

#fig.delaxes(axes[1,3])
fig.supylabel('Density')
plt.tight_layout()
plt.show()

# FUQ

In [ ]:
chey = np.concatenate((hey1[::100], hey2[::100]))

In [ ]:
res = []
for i in range(2000):
    # pesky bug was here
    res_lo = reg_lo.predict([chey[-i]])[0]
    res_hi = reg_hi.predict([chey[-i]])[0]
    res.append([res_lo, res_hi])

x = np.array(res)[:, 0]
y = np.array(res)[:, 1]

#plt.hist2d(x, y)
#sns.kdeplot(x=x, y=y, fill=True, cmap='Purples', cbar=True)
sns.jointplot(x=x, y=y, kind='kde', fill=True, cmap='Purples')

plt.xlim([0, 20])
plt.ylim([20, 45])

plt.show()

In [ ]:
mus = [10.7, 32]
sigs = [2.64, 2.64]

for mod, mu, sig in zip([reg_lo, reg_hi], mus, sigs):
    res = []
    for i in range(2000):
        res.append(mod.predict([chey[-i]])[0])
        
    c = norm(mu, sig)
    x = np.linspace(mu - 4*sig, mu + 4*sig, 100)
    y = c.pdf(x)
    plt.plot(x, y, 'r', label='Obs. with noise')
    plt.fill_between(x, y, color='r', alpha=0.5)
    
    sns.histplot(res, binwidth=0.5,
                 ec='k', stat='density', label='Forward propagation')

    plt.xlim([mu - 4*sig, mu + 4*sig])
    
    plt.xlabel('Fuel Swelling (%)')
    plt.legend()
    plt.show()

# Changes in range

In [ ]:
sns.histplot(y_train_lo, binwidth=0.5,
             stat='density', label='before')

res = []
for i in range(2000):
    res_lo = reg_lo.predict([chey[-i]])[0]
    res.append(res_lo)

sns.histplot(res, alpha=0.7, stat='density', label='after')

plt.legend()
plt.show()

In [ ]:
sns.histplot(y_train_hi, binwidth=2.5,
             stat='density', label='before')

res = []
for i in range(2000):
    res_hi = reg_hi.predict([chey[-i]])[0]
    res.append(res_hi)

sns.histplot(res, binwidth=1,
             alpha=0.7, stat='density', label='after')

plt.legend()
plt.show()

# Here we go

In [ ]:
def swelling_perc(fd):
    return 3.83e-43 * fd**2 + 4.54e-21 * fd

In [ ]:
# experiment
fdVals = list(range(8))
y_rob = [swelling_perc(x*1e21) for x in fdVals]

plt.plot(fdVals, y_rob, label='Robinson')
rob_hi = [y + 5.28 for y in y_rob]
rob_lo = [y - 5.28 for y in y_rob]
plt.fill_between(fdVals, rob_lo, rob_hi, alpha=0.5)

# prior
fd_two = [2.3, 5.3]
y_pri_lo_m = mean(y_train_lo)
y_pri_lo_d = stdev(y_train_lo)
y_pri_hi_m = mean(y_train_hi)
y_pri_hi_d = stdev(y_train_hi)
bot = [y_pri_lo_m - 2 * y_pri_lo_d, y_pri_hi_m - 2 * y_pri_hi_d]
top = [y_pri_lo_m + 2 * y_pri_lo_d, y_pri_hi_m + 2 * y_pri_hi_d]

plt.plot(fd_two, [y_pri_lo_m, y_pri_hi_m], marker='o')
plt.fill_between(fd_two, bot, top, alpha=0.6)

# posterior
res = []
for i in range(2000):
    # pesky bug was here
    res_lo = reg_lo.predict([chey[-i]])[0]
    res_hi = reg_hi.predict([chey[-i]])[0]
    res.append([res_lo, res_hi])

y_post_lo = np.array(res)[:, 0]
y_post_hi = np.array(res)[:, 1]

y_post_lo_m = mean(y_post_lo)
y_post_lo_d = stdev(y_post_lo)
y_post_hi_m = mean(y_post_hi)
y_post_hi_d = stdev(y_post_hi)
bot = [y_post_lo_m - 2 * y_post_lo_d, y_post_hi_m - 2 * y_post_hi_d]
top = [y_post_lo_m + 2 * y_post_lo_d, y_post_hi_m + 2 * y_post_hi_d]

plt.plot(fd_two, [y_post_lo_m, y_post_hi_m], marker='s')
plt.fill_between(fd_two, bot, top, alpha=0.5)

plt.xlim([2, 6])

plt.show()

# Sensitivity analysis

In [ ]:
from SALib.analyze.sobol import analyze
from SALib.sample.sobol import sample

In [ ]:
problem = {
    'num_vars': 9,
    'names': col_names,
    'bounds': [[0, 1]] * 9
}

In [ ]:
param_vals = sample(problem, 1024, calc_second_order=False)
Y = reg_lo.predict(param_vals)

Si = analyze(problem, Y,
             calc_second_order=False, print_to_console=True)

In [ ]:
Si.plot()

In [ ]:
param_vals = sample(problem, 1024, calc_second_order=False)
Y = reg_hi.predict(param_vals)

Si = analyze(problem, Y,
             calc_second_order=False, print_to_console=True)

In [ ]:
Si.plot()